In [1]:
#Focus: maintenance logging, scheduling, and resolution workflow
from datetime import datetime, timedelta, timezone
import pandas as pd
import requests
from sqlalchemy import Select
from src.database import SessionLocal, init_database
from src.models import Maintenance, Property, Tenant

init_database()

In [2]:
print(
    """
    Maintenance Requests Schema:
    -id: unique identifier
    -tenant_id: FK to Tenant
    -property_id: FK to Property
    -issue_title: short issue summary
    -issue_description: full details
    -status: ope | in_progress | scheduled | resolved | escalated
    -priority: low | medium | high | urgent
    -scheduled_for: datetime for planned visit
    -resolved_at: datetime when issue was resolved
    -notes: operational comments
    -created_at, updated_at: timestamps
    """
)


    Maintenance Requests Schema:
    -id: unique identifier
    -tenant_id: FK to Tenant
    -property_id: FK to Property
    -issue_title: short issue summary
    -issue_description: full details
    -status: ope | in_progress | scheduled | resolved | escalated
    -priority: low | medium | high | urgent
    -scheduled_for: datetime for planned visit
    -resolved_at: datetime when issue was resolved
    -notes: operational comments
    -created_at, updated_at: timestamps
    


In [3]:
from sqlalchemy import select


#CRUD helpers
def add_maintenance_request(
        tenant_id: int,
        property_id: int,
        issue_title: str,
        issue_description: str | None = None,
        status: str = "open",
        priority: str = "medium",
        notes: str = None,
):
    request = Maintenance(
        tenant_id=tenant_id,
        property_id=property_id,
        issue_title=issue_title,
        issue_description=issue_description,
        status=status,
        priority=priority,
        notes=notes,
    )

    with SessionLocal() as session:
        session.add(request)
        session.commit()
        session.refresh(request)
    return request

def get_request_by_id(request_id: int):
    with SessionLocal() as session:
        return session.get(Maintenance, request_id)

def list_maintenance_request(
        status: str | None = None,
        priority: str | None = None,
        tenant_id: int | None = None,
):
    query = select(Maintenance)

    if status:
        query = query.where(Maintenance.status == status)
    if priority:
        query = query.where(Maintenance.priority == priority)
    if tenant_id:
        query = query.where(Maintenance.tenant_id == tenant_id)

    with SessionLocal() as session:
        return session.scalars(query.order_by(Maintenance.created_at.desc())).all()

In [4]:
#Workflow helpers
def schedule_request(request_id: int, scheduled_for: datetime, notes: str | None = None):
    with SessionLocal() as session:
        request = session.get(Maintenance, request_id)
        if not request:
            return None

        request.scheduled_for = scheduled_for
        request.status = "scheduled"

        if notes:
            request.notes = f"{request.notes}\n{notes}".strip() if request.notes else notes

        session.commit()
        session.refresh(request)
        return request
def mark_in_progress(request_id: int, notes: str | None = None):
    with SessionLocal() as session:
        request = session.get(Maintenance, request_id)
        if not request:
            return None

        request.status = "in_progress"
        if notes:
            request.notes = f"{request.notes}\n{notes}".strip() if request.notes else notes

        session.commit()
        session.refresh(request)
        return request

def resolve_request(request_id: int, resolution_notes: str | None = None):
    with SessionLocal() as session:
        request = session.get(Maintenance, request_id)
        if not request:
            return None

        request.status = "resolved"
        request.resolved_at = datetime.now(timezone.utc)

        if resolution_notes:
            request.notes = (
                f"{request.notes}\nResolved: {resolution_notes}".strip()
                if request.notes
                else f"Resolved: {resolution_notes}"
            )

        session.commit()
        session.refresh(request)
        return request


def escalate_old_open_requests(days_open: int = 3):
    threshold = datetime.now(timezone.utc) - timedelta(days=days_open)

    with SessionLocal() as session:
        open_requests = session.scalars(
            select(Maintenance).where(
                Maintenance.status.in_(["open", "scheduled", "in_progress"])
            )
        ).all()

        escalated = []
        for request in open_requests:
            created = request.created_at
            if created and created.tzinfo is None:
                created = created.replace(tzinfo=timezone.utc)

            if created and created <= threshold and request.priority in ["high", "urgent"]:
                request.status = "escalated"
                request.notes = (
                    f"{request.notes}\nAuto-escalated after {days_open} days."
                    if request.notes
                    else f"Auto-escalated after {days_open} days."
                )
                escalated.append(request)

        session.commit()
        for request in escalated:
            session.refresh(request)

        return escalated

In [5]:
#reporting helpers
def maintenance_summary():
    with SessionLocal() as session:
        requests = session.scalars(select(Maintenance)).all()

    total = len(requests)
    open_count = sum(r.status == "open" for r in requests)
    in_progress_count = sum(r.status == "in_progress" for r in requests)
    scheduled_count = sum(r.status == "scheduled" for r in requests)
    resolved_count = sum(r.status == "resolved" for r in requests)
    escalated_count = sum(r.status == "escalated" for r in requests)

    high_priority_open = sum(
        r.status in ["open", "in_progress", "scheduled", "escalated"] and r.priority in ["high", "urgent"]
        for r in requests
    )

    return {
        "total_requests": total,
        "open_requests": open_count,
        "in_progress_requests": in_progress_count,
        "scheduled_requests": scheduled_count,
        "resolved_requests": resolved_count,
        "escalated_requests": escalated_count,
        "high_priority_requests": high_priority_open,
    }

def requests_to_dataframe(items):
    if not items:
        return pd.DataFrame(
            columns=[
                "id",
                "tenant_id",
                "property_id",
                "issue_title",
                "status",
                "priority",
                "scheduled_for",
                "resolved_at"
            ]
        )

    rows = []
    for r in items:
        rows.append(
            {
                "id": r.id,
                "tenant_id": r.tenant_id,
                "property_id": r.property_id,
                "issue_title": r.issue_title,
                "status": r.status,
                "priority": r.priority,
                "scheduled_for": r.scheduled_for,
                "resolved_at": r.resolved_at,
                "created_at": r.created_at,
            }
        )

    return pd.DataFrame(rows).sort_values(by="created_at", ascending=False)

In [6]:
from sqlalchemy import select


#Seed base entities

def ensure_seed_entities():
    with SessionLocal() as session:
        tenant_1 = session.get(Tenant, 1)
        if not tenant_1:
            tenant_1 = Tenant(
                id=1,
                full_name="Bennet Dyani",
                email="bennet@gmail.com",
                phone="+27630632730",
                unit_number="A1",
            )
            session.add(tenant_1)

        tenant_2 = session.get(Tenant, 2)
        if not tenant_2:
            tenant_2 = Tenant(
                id=2,
                full_name="Itumeleng Bedesho",
                email="itumeleng@gmail.com",
                phone="+27678238210",
                unit_number="B2",
            )
            session.add(tenant_2)

        property_1 = session.get(Property, 1)
        if not property_1:
            property_1 = Property(
                id=1,
                name="Sunrise Apartments",
                address="12 Main St",
                city="Cape Town",
                country="ZA",
            )
            session.add(property_1)

        property_2 = session.get(Property, 2)
        if not property_2:
            property_2 = Property(
                id=2,
                name="Green Valley Estate",
                address="45 Park Ave",
                city="Johannesburg",
                country="ZA",
            )
            session.add(property_2)

        session.commit()


def create_request_if_missing(
    tenant_id: int,
    property_id: int,
    issue_title: str,
    issue_description: str,
    priority: str,
):
    with SessionLocal() as session:
        existing = session.scalar(
            select(Maintenance).where(
                Maintenance.tenant_id == tenant_id,
                Maintenance.property_id == property_id,
                Maintenance.issue_title == issue_title,
                Maintenance.status != "resolved",
            )
        )

        if existing:
            return existing

    return add_maintenance_request(
        tenant_id=tenant_id,
        property_id=property_id,
        issue_title=issue_title,
        issue_description=issue_description,
        priority=priority,
    )


ensure_seed_entities()

req1 = create_request_if_missing(
    tenant_id=1,
    property_id=1,
    issue_title="Leaking kitchen sink",
    issue_description="Water is dripping continuously under the sink cabinet.",
    priority="high",
)

req2 = create_request_if_missing(
    tenant_id=2,
    property_id=2,
    issue_title="Bedroom light not working",
    issue_description="Ceiling light stopped working even after replacing the bulb.",
    priority="medium",
)

req3 = create_request_if_missing(
    tenant_id=1,
    property_id=1,
    issue_title="Front door lock jam",
    issue_description="Door lock occasionally jams; tenant gets locked out.",
    priority="urgent",
)

print("Seed maintenance requests ready:")
print(f"- #{req1.id}: {req1.issue_title} ({req1.priority}, {req1.status})")
print(f"- #{req2.id}: {req2.issue_title} ({req2.priority}, {req2.status})")
print(f"- #{req3.id}: {req3.issue_title} ({req3.priority}, {req3.status})")


Seed maintenance requests ready:
- #1: Leaking kitchen sink (high, open)
- #2: Bedroom light not working (medium, open)
- #3: Front door lock jam (urgent, open)


In [7]:
#Schedule

progressed = mark_in_progress(req2.id, notes="Electrician assigned to fix the light.")

resolved = resolve_request(
    req2.id,
    resolution_notes="Faulty light fixed; light is now working."
)

scheduled = schedule_request(
     req1.id,
     datetime.now(timezone.utc) + timedelta(days=1),
     notes="Plumber booked for tomorrow 10:00.",
 )

escalated = escalate_old_open_requests(days_open=0)

print("Workflow updates:")
print(f"- Scheduled request #{scheduled.id}: {scheduled.status} @ {scheduled.scheduled_for}")
print(f"- In progress request #{progressed.id}: {progressed.status}")
print(f"- Resolved request #{resolved.id}: {resolved.status} at {resolved.resolved_at}")
print(f"- Escalated count: {len(escalated)}")


Workflow updates:
- Scheduled request #1: scheduled @ 2026-07-07 08:10:01.101262+00:00
- In progress request #2: in_progress
- Resolved request #2: resolved at 2026-07-06 08:10:01.088052+00:00
- Escalated count: 2


In [8]:
#Dashboard view

all_requests = list_maintenance_request()
summary = maintenance_summary()

print("Maintenance Summary:")

for key, value in summary.items():
    print(f"{key}: {value}")

display(requests_to_dataframe(all_requests))

Maintenance Summary:
total_requests: 3
open_requests: 0
in_progress_requests: 0
scheduled_requests: 0
resolved_requests: 1
escalated_requests: 2
high_priority_requests: 2


,id,tenant_id,property_id,issue_title,status,priority,scheduled_for,resolved_at,created_at
0,3,1,1,Front door lock jam,escalated,urgent,NaT,NaT,2026-07-06 08:10:00.871903+00:00
1,2,2,2,Bedroom light not working,resolved,medium,NaT,2026-07-06 08:10:01.088052+00:00,2026-07-06 08:10:00.852970+00:00
2,1,1,1,Leaking kitchen sink,escalated,high,2026-07-07 08:10:01.101262+00:00,NaT,2026-07-06 08:10:00.822775+00:00
